# 04 - Price predictor
Estimates a price range for a job from the customer's description and the group the classifier picked.

Inputs: `uc.csv` and `service_groups.csv`. Prices are modelled in rupees, then converted to KSh at prediction time (see section 6).

Install once if needed: `pip install scikit-learn pandas joblib sentence-transformers`

In [23]:
import pandas as pd, numpy as np, re, joblib, warnings
from sklearn.linear_model import Ridge
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix
warnings.filterwarnings('ignore')

raw = pd.read_csv('data/uc.csv')
groups = pd.read_csv('data/service_groups.csv')
TRADE_MAP = {'electricians': 'electrician', 'plumbers': 'plumber', 'ac_service_repair': 'appliance',
             'ac_service_repairs': 'appliance', 'microwave_repair': 'appliance'}
df = raw.copy()
df['trade'] = df['service'].map(TRADE_MAP)
norm = lambda s: s.str.strip().str.replace(r'\s+', ' ', regex=True).str.lower()
df['name_key'] = norm(df['subservice_name'])
df['total_price'] = df['subservice_charge'].apply(lambda s: sum(int(n) for n in re.findall(r'₹(\d+)', s)))
df = df.drop_duplicates(['trade', 'name_key', 'city_name', 'subservice_charge'])
groups['name_key'] = norm(groups['name'])
m = df.merge(groups[['trade', 'name_key', 'group']], on=['trade', 'name_key'], how='inner')   # keeps only the services you approved
print(len(m), 'price rows |', m.groupby(['trade', 'name_key']).ngroups, 'services |', m.city_name.nunique(), 'cities')

7269 price rows | 193 services | 43 cities


## 1. Does city matter?
Compare how much one service's price varies across cities with how much prices vary between services.

In [24]:
m['log_price'] = np.log(m['total_price'])
m['dev_from_service'] = m['log_price'] - m.groupby(['trade', 'name_key'])['log_price'].transform('median')
svc = m.groupby(['trade', 'name_key', 'group'])['log_price'].median().reset_index()
print('spread across cities, same service (log sd):', round(m['dev_from_service'].std(), 3))
print('spread between services (log sd):          ', round(svc['log_price'].std(), 3))

spread across cities, same service (log sd): 0.107
spread between services (log sd):           1.11


City explains very little, so the model uses one national price per service (the median across cities) and ignores city. One row per service: **193 rows** to learn from, which is small, so we use a simple regularised model and cross-validate.

In [25]:
svc['price_inr'] = np.exp(svc['log_price'])
svc = svc.rename(columns={'name_key': 'service_text'})
svc.sort_values('price_inr').head(3), svc.sort_values('price_inr').tail(3)

(           trade                  service_text                group  \
 85   electrician  anchor switch (penta design)  switch_socket_board   
 54     appliance       pin valve for window ac    ac_gas_compressor   
 141  electrician      tibcon 2.5 mfd capacitor   fan_repair_install   
 
      log_price  price_inr  
 85    3.555348       35.0  
 54    3.912023       50.0  
 141   4.077537       59.0  ,
         trade          service_text              group  log_price  price_inr
 14  appliance  compressor 0.8-1 ton  ac_gas_compressor   8.779557     6500.0
 15  appliance    compressor 1.5 ton  ac_gas_compressor   8.922658     7500.0
 16  appliance      compressor 2 ton  ac_gas_compressor   9.126959     9200.0)

## 2. Features
Text of the service (turned into a multilingual embedding so it also works on Swahili or mixed customer text) plus the group as one-hot columns.

In [26]:
MODEL_NAME = 'intfloat/multilingual-e5-small'; PREFIX = 'query: '
USE_EMB = False
try:
    from sentence_transformers import SentenceTransformer
    encoder = SentenceTransformer(MODEL_NAME)
    embed = lambda texts: encoder.encode([PREFIX + t for t in texts], normalize_embeddings=True, show_progress_bar=False)
    USE_EMB = True
except Exception as e:
    print('Embedding model not available, using TF-IDF for testing only:', str(e)[:120])

group_list = sorted(svc['group'].unique())
def group_onehot(gs): return np.array([[g == k for k in group_list] for g in gs], dtype=float)

class TextFeat:
    def fit(self, texts):
        if not USE_EMB: self.v = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), sublinear_tf=True).fit(texts)
        return self
    def transform(self, texts):
        return embed(list(texts)) if USE_EMB else self.v.transform(texts).toarray()

def build(tf, texts, gs): return np.hstack([tf.transform(texts), group_onehot(gs)])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2693.73it/s]


## 3. Cross-validated comparison
Five folds over **services**, so every score is on services the model has not seen. Baselines: one global median, and the median of the group.

In [27]:
y = svc['log_price'].values; n = len(svc)
rng = np.random.RandomState(0); folds = np.array_split(rng.permutation(n), 5)
ALPHA = 1.0
def cv_predictions(alpha):
    p_model = np.zeros(n); p_group = np.zeros(n); p_glob = np.zeros(n)
    for te in folds:
        tr = np.setdiff1d(np.arange(n), te); st, se = svc.iloc[tr], svc.iloc[te]
        tf = TextFeat().fit(st.service_text)
        r = Ridge(alpha=alpha).fit(build(tf, st.service_text, st.group), y[tr])
        p_model[te] = r.predict(build(tf, se.service_text, se.group))
        gm = st.groupby('group')['log_price'].median()
        p_glob[te] = np.median(y[tr]); p_group[te] = se['group'].map(gm).fillna(np.median(y[tr])).values
    return p_model, p_group, p_glob

def report(name, p):
    err = np.abs(np.exp(p) / np.exp(y) - 1)
    print(f'{name:18s} MAE(log) {np.mean(np.abs(y - p)):.3f} | median error {np.median(err)*100:.0f}% | within 50%: {np.mean(err <= .5)*100:.0f}%')

scores = {}
for a in [0.001, 0.01, 0.1, 1, 10]:
    pm, pg, pgl = cv_predictions(a); scores[a] = np.mean(np.abs(y - pm)); print('alpha', a); report('  ridge', pm)
ALPHA = min(scores, key=scores.get)     # best alpha by cross-validated error
print('using alpha', ALPHA, '(chosen on the same folds, so the score below is slightly optimistic)')
p_model, p_group, p_glob = cv_predictions(ALPHA)
print(); report('global median', p_glob); report('group median', p_group); report('ridge text+group', p_model)

alpha 0.001
  ridge            MAE(log) 0.573 | median error 44% | within 50%: 56%
alpha 0.01
  ridge            MAE(log) 0.493 | median error 37% | within 50%: 63%
alpha 0.1
  ridge            MAE(log) 0.513 | median error 43% | within 50%: 57%
alpha 1
  ridge            MAE(log) 0.612 | median error 48% | within 50%: 52%
alpha 10
  ridge            MAE(log) 0.760 | median error 60% | within 50%: 39%
using alpha 0.01 (chosen on the same folds, so the score below is slightly optimistic)

global median      MAE(log) 0.907 | median error 65% | within 50%: 34%
group median       MAE(log) 0.718 | median error 57% | within 50%: 44%
ridge text+group   MAE(log) 0.493 | median error 37% | within 50%: 63%


## 4. Price ranges
A single number would be misleading, so the app shows a range. The range is the prediction plus or minus a margin taken from the cross-validated errors, using the margin from the *other* folds so the coverage check is honest.

In [28]:
resid = np.abs(y - p_model)
fold_id = np.zeros(n, int)
for k, te in enumerate(folds): fold_id[te] = k
cover = []
for k in range(5):
    q = np.quantile(resid[fold_id != k], 0.8)
    te = fold_id == k
    cover.append(np.mean(np.abs(y[te] - p_model[te]) <= q))
print('80% range covers', round(np.mean(cover) * 100), '% of held-out services (target 80%)')
MARGIN = float(np.quantile(resid, 0.8))
print('margin: x', round(np.exp(MARGIN), 2), 'either side of the estimate')

80% range covers 80 % of held-out services (target 80%)
margin: x 2.16 either side of the estimate


## 5. Fit the final model on all services and define the estimator

In [29]:
tf_final = TextFeat().fit(svc.service_text)
ridge_final = Ridge(alpha=ALPHA).fit(build(tf_final, svc.service_text, svc.group), y)

def estimate_price_inr(text, group):
    x = build(tf_final, [text.lower()], [group])
    mid = float(np.exp(ridge_final.predict(x)[0]))
    return {'low': mid / np.exp(MARGIN), 'mid': mid, 'high': mid * np.exp(MARGIN)}

for t, g in [('kitchen tap leaking', 'tap_mixer'), ('change a bulb', 'lights_bulbs'), ('install ceiling fan', 'fan_repair_install')]:
    print(t, '->', {k: round(v) for k, v in estimate_price_inr(t, g).items()}, 'INR')

kitchen tap leaking -> {'low': 49, 'mid': 107, 'high': 231} INR
change a bulb -> {'low': 43, 'mid': 94, 'high': 202} INR
install ceiling fan -> {'low': 66, 'mid': 143, 'high': 308} INR


## 6. Rupees to shillings
`KES_PER_INR` is the starting exchange rate. Update it to the live rate on the day you run this. When you have real Nairobi quotes, save them as `nairobi_quotes.csv` with columns `service_text, price_kes` (`service_text` must match a catalogue service name, lowercase) and run `calibrate('data/nairobi_quotes.csv')`. By default it fits a power law, KSh = a x INR^b, which allows a minimum charge on cheap jobs and slower growth on expensive ones. Use `method='ratio'` for a single multiplier. Per-trade ratios are only used when a trade has at least `min_quotes` quotes (default 15). Section 6b checks which method predicts held-out quotes better.

In [30]:
KES_PER_INR = 1.37       # starting point only; check the live INR to KES rate
COEF = {'default': KES_PER_INR, 'power': None}

def calibrate(path='nairobi_quotes.csv', method='power', min_quotes=15):
    q = pd.read_csv(path)[['service_text', 'price_kes']]; q['service_text'] = norm(q['service_text'])   # extra columns are ignored
    j = q.merge(svc[['service_text', 'trade', 'price_inr']], on='service_text')
    j['ratio'] = j['price_kes'] / j['price_inr']
    print(len(j), 'of', len(q), 'quotes matched a catalogue service; median ratio', round(j['ratio'].median(), 2))
    COEF['default'] = float(j['ratio'].median())
    if method == 'power':
        # KSh = a * INR^b : captures a minimum charge on cheap jobs and slower growth on expensive ones
        b, la = np.polyfit(np.log(j['price_inr']), np.log(j['price_kes']), 1)
        # error of the conversion itself: leave-one-out residuals on the log scale
        xs, ys = np.log(j['price_inr'].values), np.log(j['price_kes'].values); loo = []
        for i in range(len(j)):
            keep = np.arange(len(j)) != i
            bb, aa = np.polyfit(xs[keep], ys[keep], 1); loo.append(abs(aa + bb * xs[i] - ys[i]))
        COEF['power'] = {'a': float(np.exp(la)), 'b': float(b), 'conv_margin': float(np.quantile(loo, 0.8))}
        print(f"power-law fit: KSh = {np.exp(la):.1f} x INR^{b:.2f}")
    for t, d in j.groupby('trade'):          # per-trade ratios only with plenty of quotes
        if len(d) >= min_quotes: COEF[t] = float(d['ratio'].median())
    return COEF

TRADE_OF = svc.drop_duplicates('group').set_index('group')['trade'].to_dict()
def convert(inr, group):
    if COEF.get('power'): return COEF['power']['a'] * inr ** COEF['power']['b']
    return inr * COEF.get(TRADE_OF[group], COEF['default'])

def estimate_price_kes(text, group):
    r = estimate_price_inr(text, group)
    p = COEF.get('power')
    if p:
        mid = p['a'] * r['mid'] ** p['b']
        # range = price-model error (scaled by the exponent) combined with the conversion error
        M = np.sqrt((p['b'] * MARGIN) ** 2 + p['conv_margin'] ** 2)
        out = {'low': mid / np.exp(M), 'mid': mid, 'high': mid * np.exp(M)}
    else:
        out = {k: convert(v, group) for k, v in r.items()}
    return {k: int(round(v, -1)) for k, v in out.items()}

print(estimate_price_kes('kitchen tap leaking', 'tap_mixer'))

{'low': 70, 'mid': 150, 'high': 320}


In [31]:
calibrate('data/nairobi_quotes.csv')

26 of 26 quotes matched a catalogue service; median ratio 4.0
power-law fit: KSh = 48.7 x INR^0.56


{'default': 4.0040080160320635,
 'power': {'a': 48.667738523653725,
  'b': 0.5645076407110122,
  'conv_margin': 0.8921709340103829}}

## 6b. Check the calibration
Leave-one-out: predict each quote from a fit on all the others, and compare a single ratio, the power law and the plain exchange rate. Run after `calibrate(...)`.

In [32]:
def calibration_check(path='nairobi_quotes.csv'):
    q = pd.read_csv(path)[['service_text', 'price_kes']]; q['service_text'] = norm(q['service_text'])
    j = q.merge(svc[['service_text', 'price_inr']], on='service_text').reset_index(drop=True)
    x, yk = np.log(j['price_inr'].values), np.log(j['price_kes'].values)
    res = {'plain exchange rate': [], 'single ratio': [], 'power law': []}
    for i in range(len(j)):
        m = np.arange(len(j)) != i
        preds = {'plain exchange rate': x[i] + np.log(KES_PER_INR),
                 'single ratio': x[i] + np.median(yk[m] - x[m]),
                 'power law': np.polyval(np.polyfit(x[m], yk[m], 1), x[i])}
        for k, p in preds.items(): res[k].append(abs(np.exp(p - yk[i]) - 1))
    for k, e in res.items():
        print(f'{k:20s} median error {np.median(e)*100:.0f}% | within 50%: {np.mean(np.array(e) <= .5)*100:.0f}%')

calibration_check('data/nairobi_quotes.csv')

plain exchange rate  median error 68% | within 50%: 31%
single ratio         median error 51% | within 50%: 50%
power law            median error 39% | within 50%: 65%


## 7. Save

In [33]:
joblib.dump({'ridge': ridge_final, 'tfidf': getattr(tf_final, 'v', None), 'use_emb': USE_EMB,
             'model_name': MODEL_NAME, 'prefix': PREFIX, 'margin': MARGIN, 'coef': COEF,
             'groups': group_list, 'trade_of': TRADE_OF}, 'price_model.joblib')
print('saved price_model.joblib')

saved price_model.joblib
